# Step 5 — HfO2 and ZrO2 polymorphs with universal MLIPs
Phases: monoclinic P2_1/c (ground state), **polar orthorhombic Pca2_1 (the ferroelectric phase)**, antipolar orthorhombic Pbca, tetragonal P4_2/nmc, cubic Fm-3m.
Starting structures and reference PBE energies come from the **Materials Project** (free account; you need your personal API key — it is asked in a hidden box and is not saved in the notebook. Never share it).

**Run on Colab with GPU T4**, once per family: set `FAMILY` in cell 1 (`"mace"`, `"chgnet"`, `"sevennet"`, `"mattersim"`, `"orb"`), Exécution → Tout exécuter, then disconnect/delete the runtime before the next family. Each run downloads `step5_<family>.csv` (and the first run also `step5_mp_reference.csv`).

In [ ]:
FAMILY = "orb"     # run 5 times: "mace", "chgnet", "sevennet", "mattersim", "orb"
PKG = {"mace": "mace-torch", "chgnet": "chgnet", "sevennet": "sevenn", "mattersim": "mattersim", "orb": "orb-models"}[FAMILY]
if FAMILY == "orb":   # orb-models pins an old dm-tree without Python 3.13 wheels
    !pip install -q "dm-tree>=0.1.9" cached_path phonopy spglib "ase>=3.23" mp-api pymatgen
    !pip install -q --no-deps "orb-models>=0.5,<0.6"
    import importlib.metadata as _md, subprocess as _sp
    _reqs = [r.split(";")[0].strip() for r in (_md.requires("orb-models") or [])
             if "extra ==" not in r and not r.lower().replace("_", "-").startswith("dm-tree")]
    print("orb-models", _md.version("orb-models"), "| installing dependencies:", _reqs)
    _sp.run(["pip", "install", "-q", *_reqs])
else:
    !pip install -q {PKG} phonopy spglib "ase>=3.23" mp-api pymatgen


In [ ]:
# Materials Project structures (API key typed in a hidden box)
import getpass, numpy as np, pandas as pd
from mp_api.client import MPRester
from pymatgen.io.ase import AseAtomsAdaptor
API_KEY = getpass.getpass("Materials Project API key (hidden): ")
WANT = ["P2_1/c", "Pca2_1", "Pbca", "P4_2/nmc", "Fm-3m"]
STRUCT, mprow = {}, []
with MPRester(API_KEY) as mpr:
    for f in ["HfO2", "ZrO2"]:
        docs = mpr.materials.summary.search(formula=f, fields=["material_id", "symmetry", "structure",
                       "energy_above_hull", "uncorrected_energy_per_atom", "nsites"])
        for sg in WANT:
            c = [d for d in docs if d.symmetry.symbol == sg]
            if not c: print("not found:", f, sg); continue
            d = min(c, key=lambda d: d.uncorrected_energy_per_atom)
            STRUCT[(f, sg)] = AseAtomsAdaptor.get_atoms(d.structure.get_primitive_structure())
            mprow.append(dict(material=f, phase=sg, mp_id=str(d.material_id),
                              E_mp_eV_per_atom=d.uncorrected_energy_per_atom, nsites=len(STRUCT[(f, sg)])))
mp = pd.DataFrame(mprow)
e0 = mp[mp.phase == "P2_1/c"].set_index("material").E_mp_eV_per_atom
mp["dE_mp_meV_per_fu"] = 3000 * (mp.E_mp_eV_per_atom - mp.material.map(e0))
del API_KEY
mp.round(4)


In [ ]:
import torch, time, spglib, ase, importlib.metadata as md
from ase.constraints import FixSymmetry
from ase.filters import FrechetCellFilter
from ase.optimize import BFGS
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "|", PKG, md.version(PKG))
MODELS = {"mace": ["small", "medium", "large", "medium-mpa-0", "medium-omat-0", "mace-matpes-pbe-0", "mace-matpes-r2scan-0"],
          "chgnet": ["chgnet-0.3.0"],
          "sevennet": ["7net-0", "7net-mf-ompa"],
          "mattersim": ["MatterSim-v1.0.0-1M", "MatterSim-v1.0.0-5M"],
          "orb": ["orb-v3-conservative-inf-mpa", "orb-v3-conservative-inf-omat"]}[FAMILY]

def get_calc(name):
    if FAMILY == "mace":
        from mace.calculators import mace_mp
        return mace_mp(model=name, device=device, default_dtype="float64")
    if FAMILY == "chgnet":
        from chgnet.model import CHGNet
        from chgnet.model.dynamics import CHGNetCalculator
        return CHGNetCalculator(CHGNet.load(model_name=name.split("-")[1]), use_device=device)
    if FAMILY == "sevennet":
        from sevenn.calculator import SevenNetCalculator
        if name == "7net-mf-ompa":
            return SevenNetCalculator(model=name, modal="mpa", device=device)
        return SevenNetCalculator(model=name, device=device)
    if FAMILY == "mattersim":
        from mattersim.forcefield import MatterSimCalculator
        return MatterSimCalculator(load_path=name + ".pth", device=device)
    if FAMILY == "orb":
        import pkgutil, importlib, inspect, orb_models
        from orb_models.forcefield import pretrained
        # find ORBCalculator wherever this version of orb-models puts it
        Calc = None
        for mi in pkgutil.walk_packages(orb_models.__path__, "orb_models."):
            if "calculator" not in mi.name: continue
            try: mod = importlib.import_module(mi.name)
            except Exception as ex: print("cannot import", mi.name, repr(ex)[:200]); continue
            if hasattr(mod, "ORBCalculator"): Calc = mod.ORBCalculator; print("ORBCalculator from", mi.name); break
        assert Calc is not None, "ORBCalculator not found"
        fname = name.replace("-", "_")
        avail = [f for f in dir(pretrained) if f.startswith("orb")]
        print("available ORB models:", avail)
        assert fname in avail, f"{fname} not in this orb-models version"
        fn = getattr(pretrained, fname)
        kw = {"device": device}
        if "precision" in inspect.signature(fn).parameters: kw["precision"] = "float32-highest"
        orbff = fn(**kw)
        ckw = {"device": device} if "device" in inspect.signature(Calc).parameters else {}
        return Calc(orbff, **ckw)
SYSTEMS = {"BaTiO3": ("Ba","Ti"), "PbTiO3": ("Pb","Ti"), "KNbO3": ("K","Nb"),
           "SrTiO3": ("Sr","Ti"), "KTaO3": ("K","Ta")}
DIRS = {"Pm-3m": None, "P4mm": [0,0,1], "Amm2": [1,1,0], "R3m": [1,1,1]}
QP = {"G": [0,0,0], "X": [0,0,.5], "M": [.5,.5,0], "R": [.5,.5,.5]}
THz2cm = 33.35641

def spg(at, prec=1e-3):
    return spglib.get_spacegroup((at.cell[:], at.get_scaled_positions(), at.numbers), symprec=prec)

def relax(at, calc, fmax=1e-3, steps=3000):
    at = at.copy(); at.calc = calc
    at.set_constraint(FixSymmetry(at, symprec=1e-3))
    opt = BFGS(FrechetCellFilter(at), logfile=None); conv = opt.run(fmax=fmax, steps=steps)
    at.set_constraint(); return at, bool(conv), opt.nsteps

def run_model(MODEL):
    calc = get_calc(MODEL); rows = []
    for (f, sg), at0 in STRUCT.items():
        t0 = time.time(); at, conv, n = relax(at0, calc)
        nfu = len(at) // 3; cp = at.cell.cellpar()
        rows.append(dict(model=MODEL, material=f, phase=sg, E_eV_per_fu=at.get_potential_energy() / nfu,
                         vol_per_fu=at.get_volume() / nfu, a=cp[0], b=cp[1], c=cp[2], alpha=cp[3], beta=cp[4], gamma=cp[5],
                         spg_start=spg(at0), spg_final=spg(at), converged=conv, steps=n, time_s=round(time.time() - t0, 1)))
        print(f"{MODEL:28s} {f} {sg:9s} {rows[-1]['spg_final']:14s} conv={conv}")
    df = pd.DataFrame(rows)
    e0 = df[df.phase == "P2_1/c"].set_index("material").E_eV_per_fu
    df["dE_meV_per_fu"] = 1000 * (df.E_eV_per_fu - df.material.map(e0))
    return df


In [ ]:
all_df, failed = [], {}
for m in MODELS:
    try:
        t0 = time.time(); all_df.append(run_model(m)); print(f"--- {m} done in {time.time()-t0:.0f} s")
    except Exception as e:
        failed[m] = repr(e); print("FAILED", m, repr(e)[:300])
    torch.cuda.empty_cache()
print("failed:", failed)
assert all_df, "all models failed - send the FAILED lines to Claude"
res = pd.concat(all_df, ignore_index=True); res["package_version"] = md.version(PKG)
out = res.merge(mp, on=["material", "phase"], how="left")
out.pivot_table(index=["material", "phase"], columns="model", values="dE_meV_per_fu").round(1).join(
    mp.set_index(["material", "phase"]).dE_mp_meV_per_fu.round(1))


In [ ]:
from google.colab import files
fn = f"step5_{FAMILY}.csv"; out.to_csv(fn, index=False); files.download(fn)
mp.to_csv("step5_mp_reference.csv", index=False); files.download("step5_mp_reference.csv")
from ase.io import write as _w
_frames = []
for (f, sg), at in STRUCT.items():
    a = at.copy(); a.info["material"] = f; a.info["phase"] = sg; _frames.append(a)
_w("step5_structures.extxyz", _frames); files.download("step5_structures.extxyz")
print("saved", fn)
